In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("..")
from lens.raytrace import trace, best_spot, spot_at
from lens.design import R2_for

In [2]:
R1, t = 29.5, 6.0
R2 = R2_for(R1, t, 50)
design = [(0, R1, 1.5), (t, R2, 1.0)]
f5 = np.linspace(-5, 5, 1001)        # f/5: rays from -5 to 5

for deg in [0, 5, 10]:
    print(deg, best_spot(design, np.radians(deg), f5))

0 (np.float64(52.3), np.float64(0.00801864764938113))
5 (np.float64(51.650000000000006), np.float64(0.008317115563164479))
10 (np.float64(49.75), np.float64(0.009597322009676593))


at its own best focus, each angle is now very sharp (about 0.008 mm). 

halving the opening cut the spherical blur by about 8 times. 

but the bowl is still there: the best positions are 52.3 -> 51.65 -> 49.75. so now field curvature is the main problem.

In [3]:
def field_loss(surfaces, z_sensor, z_stop = 0.0, rays = np.linspace(-5, 5, 1001)):
    spots = []
    for deg in [0, 5, 10]:
        z_last, y, u = trace(rays, surfaces, np.radians(deg), z_stop)
        y_sensor = y + np.tan(u) * (z_sensor - z_last)
        spots.append(np.std(y_sensor))
    return np.mean(spots)

def best_field(surfaces, z_stop = 0.0):
    sensors = np.linspace(44, 56, 241)
    losses = []
    for z in sensors:
        losses.append(field_loss(surfaces, z, z_stop))
    losses = np.array(losses)
    best = np.argmin(losses)
    return sensors[best], losses[best]

In [4]:
print(best_field(design))
print(field_loss(design, 52.3))

(np.float64(51.65), np.float64(0.05449188176817022))
0.06797793310891687


at the center's best focus (52.3), the field loss is 0.068.

at the compromise (51.65), it's 0.054.

In [5]:
for R1_try in [29.5, 100]:
    lens = [(0, R1_try, 1.5), (t, R2_for(R1_try, t, 50), 1.0)]
    for z_stop in [0, -5, -10, -15, -20]:
        print(R1_try, z_stop, best_field(lens, z_stop))

29.5 0 (np.float64(51.65), np.float64(0.05449188176817022))
29.5 -5 (np.float64(51.65), np.float64(0.05458206310341553))
29.5 -10 (np.float64(51.6), np.float64(0.057676860313359106))
29.5 -15 (np.float64(51.55), np.float64(0.06353607693603831))
29.5 -20 (np.float64(51.45), np.float64(0.07202433066801976))
100 0 (np.float64(53.6), np.float64(0.06745977441360265))
100 -5 (np.float64(53.75), np.float64(0.048534061011852515))
100 -10 (np.float64(53.85), np.float64(0.035347206932210766))
100 -15 (np.float64(53.85), np.float64(0.029462392350194832))
100 -20 (np.float64(53.85), np.float64(0.03334197021075485))


this is the key result. for the best-form lens, moving the stop only makes things worse. 

for the lens bent the other way, moving the stop to −15 cuts the field loss to 0.029, almost half the best-form lens's best (0.054).

In [6]:
losses, R1s, zstops, sensors = [], [], [], []
for R1try in [29.5, 40, 60, 100, 200, 300]:
    lens1 = [(0, R1try, 1.5), (t, R2_for(R1try, t, 50), 1.0)]
    for z_stop1 in [0, -5, -10, -15, -20]:
        z_best, loss = best_field(lens1, z_stop1)
        print(R1try, z_stop1, z_best, loss)
        losses.append(loss)
        R1s.append(R1try)
        zstops.append(z_stop1)
        sensors.append(z_best)

losses = np.array(losses)
best = np.argmin(losses)
print(f"best combination:\nR1 = {R1s[best]}, R2 = {R2_for(R1s[best], t, 50)}, z_stop = {zstops[best]}, sensor = {sensors[best]}, field loss = {losses[best]}")

29.5 0 51.65 0.05449188176817022
29.5 -5 51.65 0.05458206310341553
29.5 -10 51.6 0.057676860313359106
29.5 -15 51.55 0.06353607693603831
29.5 -20 51.45 0.07202433066801976
40 0 52.5 0.05640591544134668
40 -5 52.55 0.04843031024594546
40 -10 52.6 0.043600048768936674
40 -15 52.6 0.04239446650147971
40 -20 52.6 0.04501446343005989
60 0 53.15 0.06147770763250196
60 -5 53.3 0.04716860032259226
60 -10 53.35 0.03708287322782494
60 -15 53.4 0.031985119636695265
60 -20 53.4 0.03300522101545517
100 0 53.6 0.06745977441360265
100 -5 53.75 0.048534061011852515
100 -10 53.85 0.035347206932210766
100 -15 53.85 0.029462392350194832
100 -20 53.85 0.03334197021075485
200 0 53.9 0.07323962227821469
200 -5 54.05 0.05110197892815349
200 -10 54.15 0.0363924854425274
200 -15 54.15 0.03200828857531612
200 -20 54.15 0.04059574425480189
300 0 53.95 0.07545044110100574
300 -5 54.15 0.052271169918830206
300 -10 54.25 0.037245744093329584
300 -15 54.25 0.03387860654264825
300 -20 54.2 0.044333714377195944
best c

grid search over 6 front radii × 5 stop positions (30 lenses, f = 50 mm kept fixed by R2_for). for each one, best_field picks the flat sensor position with the lowest field loss (average spot at 0°, 5° and 10°).

best combination: R1 = 100 mm, R2 = −32.67 mm, stop 15 mm in front of the lens, sensor at 53.85 mm. field loss 0.0295 mm, 46% lower than the best-form lens at its best (0.0545 mm).

the winner is bent the other way from the best-form lens: gentle front, strong back. moving the stop only helps lenses bent this way. for the best-form lens it makes things worse. the stop and the shape have to change together.

In [7]:
design = [(0, R1, 1.5), (t, R2, 1.0)]
winner = [(0, 100, 1.5), (t, R2_for(100, t, 50), 1.0)]
print("compromise of best-form lens design")
for deg in [0, 5, 10]:
    print(deg, spot_at(design, 51.65, np.radians(deg), f5))
print("best design from previous exercise")
for deg in [0, 5, 10]:
    print(deg, spot_at(winner, 53.85, np.radians(deg), f5, -15))

compromise of best-form lens design
0 0.03846901212554951
5 0.008317115563164452
10 0.1166895176157967
best design from previous exercise
0 0.028177996213383242
5 0.02225362793284609
10 0.037955552904355164


each lens measured at its own compromise sensor (best-form at 51.65 mm, winner at 53.85 mm with the stop at −15 mm), at f/5:

| angle | best-form | winner |
|---|---|---|
| 0° | 0.038 mm | 0.028 mm |
| 5° | 0.008 mm | 0.022 mm |
| 10° | 0.117 mm | 0.038 mm |

the winner has the sharper center and an edge about 3× sharper. the best-form lens only beats it at 5°, because its compromise sensor happens to sit right on the 5° focus, while 0° and 10° are badly off on opposite sides of the curve.

the winner is far more even: all three spots are within 0.022–0.038 mm, while the best-form lens ranges from 0.008 to 0.117 mm (about 15×) across one photo. for a camera, an evenly sharp image matters more than one perfect ring.

In [8]:
print("best-form lens design")
for deg in [0, 5, 10]:
    print(deg, best_spot(design, np.radians(deg), f5))
print("best design from previous exercise")
for deg in [0, 5, 10]:
    print(deg, best_spot(winner, np.radians(deg), f5, -15))

best-form lens design
0 (np.float64(52.3), np.float64(0.00801864764938113))
5 (np.float64(51.650000000000006), np.float64(0.008317115563164479))
10 (np.float64(49.75), np.float64(0.009597322009676593))
best design from previous exercise
0 (np.float64(54.150000000000006), np.float64(0.021395558623615297))
5 (np.float64(53.95), np.float64(0.021423510433932685))
10 (np.float64(53.3), np.float64(0.021516344612333215))


each angle's own best focus, at f/5:

| angle | best-form (stop at lens) | winner (stop at −15 mm) |
|---|---|---|
| 0° | 52.3 mm | 54.15 mm |
| 5° | 51.65 mm | 53.95 mm |
| 10° | 49.75 mm | 53.3 mm |
| spread | 2.55 mm | 0.85 mm |

the winner's sharp image still lies on a curve, but it is about 3× flatter. that's why one flat sensor can sit close to every angle's focus at once, and why the winner's spots are so even across the photo. moving the stop in front of the lens, together with bending the lens the other way, is what flattens it.

at each angle's own best focus, the winner's spot is about 0.021 mm, about 2.7× bigger than the best-form lens's 0.008 mm. bending the lens the other way brings back spherical aberration, which the best-form shape keeps lowest.

but those own-focus spots are only possible if the sensor could sit at every angle's focus at once, and a flat sensor can't. on one sensor, the best-form lens ranges from 0.008 to 0.117 mm across the photo, while the winner stays within 0.022–0.038 mm. we gave up some sharpness at the ideal focus to get a much flatter, more even image, which is what the sensor actually records.

(measured in 2D; a full 3D trace would also show some astigmatism, so the real gain would be smaller.)